# cross-product-normal — worked example 1: Lambertian shade of a single triangle from its normal

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-product-normal`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A triangle's unit surface normal is `cross(P2-P1, P3-P1)` normalized to length 1. Lambertian (diffuse) shading multiplies the light intensity by `max(0, normal · light_dir)`, where `light_dir` points from the surface toward the light. The dot product is largest when the face points straight at the light and clamps to 0 when the face turns away.

## Worked solution

**Step 1 — edges from the shared vertex.** We pick `P1` as the anchor and form `e1 = P2 - P1` and `e2 = P3 - P1`. Two edges that share a vertex span the plane of the triangle, so their cross product is perpendicular to that plane.

**Step 2 — cross product.** `n = t.linalg.cross(e1, e2)` is perpendicular to both edges, hence to the face. Its length equals twice the triangle area, so it is *not* unit length yet.

**Step 3 — normalize.** Dividing by `n.norm()` gives a unit normal. Shading only depends on direction, so we must remove the magnitude.

**Step 4 — normalize the light direction too.** The dot product `normal · light` equals `cos(angle)` only when *both* vectors are unit length. We divide the light vector by its norm before the dot.

**Step 5 — clamp.** A negative dot product means the face points away from the light, which physically receives no diffuse light, so we clamp with `max(0, ·)`. The result is a scalar brightness in `[0, 1]`.

In [ ]:
def lambert_shade(P1: Tensor, P2: Tensor, P3: Tensor, light: Tensor) -> Tensor:
    e1 = P2 - P1
    e2 = P3 - P1
    n = t.linalg.cross(e1, e2)
    normal = n / n.norm()
    light_dir = light / light.norm()
    return (normal @ light_dir).clamp(min=0.0)

P1 = t.tensor([0.0, 0.0, 0.0])
P2 = t.tensor([2.0, 0.0, 0.0])
P3 = t.tensor([0.0, 2.0, 0.0])
light = t.tensor([0.0, 0.0, 5.0])
print(lambert_shade(P1, P2, P3, light))